# Manual Round

You have to submit a single limit order (price, quantity). When the auction ends, the exchange selects a single clearing price that:

1. maximizes total traded volume, then
2. breaks ties by choosing the higher price.

All bids with price ≥ clearing price and asks with price ≤ clearing price execute at the clearing price. Allocation is price priority, then time priority. Since you are last to submit, you are last in line at any price level you join.

As a special welcoming event, the Intarians are organizing an exchange auction. You will have access to two stale order books: one for Dryland Flax and one for Ember Mushroom. You can put in one order per tradable good. When the auction clears, it will execute with price-time priority and there will be no more continuous trading to parttake in. How might you turn this celebratory opportunity into profit?

Note: Price-time priority is a core order-matching algorithm where exchanges fill orders based first on the best price (highest bid/lowest offer) and second on the time they were placed (FIFO - First In, First Out). This system provides fairness, as it rewards traders for posting competitive prices and being quick to fill orders.

# Dryland Flax
This order book shows a stale state of bids and asks for Dryland Flax. Use this to determine your own strategy and order input. Any Dryland Flax you acquire will be sold directly after the auction closes for **30 XIRECs** per piece.

In [ ]:
flax_bids = [
    (30, 30000),
    (29, 5000),
    (28, 12000),
    (27, 28000)
]
flax_asks = [
    (28, 40000), 
    (31, 20000), 
    (32, 20000), 
    (33, 30000)
]

mush_bids = [(20, 43000), (19, 17000), (18, 6000), (17, 5000), (16, 10000), (15, 5000), (14, 10000), (13, 7000)]
mush_asks = [(12, 20000), (13, 25000), (14, 35000), (15, 6000), (16, 5000), (18, 10000), (19, 12000)]

from dataclasses import dataclass

@dataclass
class Order:
    price: int
    qty: int
    time: int # My Time is 1, Book is 0 (book gets priority if equal prices)

def get_clearing_price(bids, asks):
    prices = sorted({o.price for o in bids + asks})
    best_vol, cp = -1, -1
    # tests prices to find the one that maximizes volume traded
    for p in prices:
        vol = min(sum(o.qty for o in bids if o.price >= p),
                  sum(o.qty for o in asks if o.price <= p))
        if vol > best_vol or (vol == best_vol and vol > 0 and p > cp):
            best_vol, cp = vol, p
    return cp

def simulate_my_pnl(my_price, my_qty, book_bids, book_asks, fv, fee):
    bids = [Order(p, q, 0) for p, q in book_bids] + [Order(my_price, my_qty, 1)] # <-- my order added to bids
    asks = [Order(p, q, 0) for p, q in book_asks]
    cp = get_clearing_price(bids, asks)

    if cp == -1 or my_price < cp:
        return 0.0, 0, cp

    supply = sum(o.qty for o in asks if o.price <= cp)
    priority = sum(o.qty for o in bids if o.price > my_price or
                   (o.price == my_price and o.time == 0))
    fill = max(0, min(my_qty, supply - priority))
    return fill * (fv - cp - fee), fill, cp

def brute_force(book_bids, book_asks, fv, fee):
    best = {"pnl": -1.0}
    prices = sorted({p for p, _ in book_bids + book_asks})

    for p in prices: # simulate p in all prices in the book
        for q in range(1, 100_001): # simulate 0<q<100001
            
            
            pnl, fill, cp = simulate_my_pnl(p, q, book_bids, book_asks, fv, fee)
            if pnl > best["pnl"]:
                best = {"pnl": round(pnl, 2), "bid_p": p, "bid_q": q, "fill": fill, "cp": cp}
                
            
    return best


print("Flax:", brute_force(flax_bids, flax_asks, 30.0, 0.0))
print("Mushroom:", brute_force(mush_bids, mush_asks, 20.0, 0.1))

Flax: {'pnl': 9999.0, 'bid_p': 30, 'bid_q': 9999, 'fill': 9999, 'cp': 29}
Mushroom: {'pnl': 77996.1, 'bid_p': 17, 'bid_q': 19999, 'fill': 19999, 'cp': 16}


# Ember Mushrooms
This order book shows a stale state of bids and asks for Ember Mushroom. Use this to determine your own strategy and order input. **Trading fees per volume apply: 0.05 XIRECs for buying and 0.05 XIRECs for selling**. Any Ember Mushroom you acquire will be sold directly after the auction closes for **20 XIRECs** per piece